# Chapter 1 assignment: a first physics-informed neural network

**Book:** *Physics-Informed Neural Networks for Ordinary, Integral and Integro-Differential Equations*, Chapter 1 (Formulation and loss design).

## Learning goals
After this assignment you will be able to

1. write the residual and the loss of a PINN for a scalar initial value problem,
2. train a PINN with soft and with hard (exact) enforcement of the initial condition using the companion library `pinnode`,
3. estimate the variance of the Monte Carlo residual loss for random collocation points, and
4. compare the PINN with a classical adaptive Runge-Kutta solver in a fair way.

Everything runs on a CPU in double precision; each training takes well under a minute with the suggested settings.

## Short theory recap

For the IVP $y'(t)=f(t,y)$, $y(t_0)=y_0$, a PINN uses a network $y_\theta(t)$ and the **residual**
$$r_\theta(t)=y_\theta'(t)-f(t,y_\theta(t)).$$
With collocation points $t_1,\dots,t_M$ the **soft** loss is
$$\mathcal{L}(\theta)=\frac1M\sum_{i=1}^M r_\theta(t_i)^2+\lambda_{ic}\,\bigl(y_\theta(t_0)-y_0\bigr)^2 .$$
The **hard** formulation uses the trial function $y_\theta(t)=y_0+(t-t_0)N_\theta(t)$, which satisfies the initial condition for every $\theta$, so that only the residual term remains.
For independent uniformly distributed points the residual term is an unbiased Monte Carlo estimate of $\frac{1}{|\Omega|}\int_\Omega r_\theta^2\,dt$ with variance $\frac1M\bigl(\frac{1}{|\Omega|}\int r^4-(\frac{1}{|\Omega|}\int r^2)^2\bigr)$ (Proposition in Section 1.5).

**Problem of this assignment:** $y'=-2ty$, $y(0)=1$ on $[0,2]$, with exact solution $y(t)=e^{-t^2}$.

In [ ]:
import time
import numpy as np
import torch
from scipy.integrate import solve_ivp

from pinnode import (MLP, ConstrainedModel, configure_torch, derivative, linf, rel_l2,
                     set_seed, to_tensor, train)

configure_torch("float64", threads=1)   # double precision, one thread (as in the book)
set_seed(1234)
T = 2.0
t_test = np.linspace(0.0, T, 1001)
y_exact = np.exp(-t_test ** 2)

## Task 1: the residual

Complete `residual(model, t)` so that it returns $r_\theta(t)=y_\theta'(t)+2t\,y_\theta(t)$ at the points `t`
(a tensor of shape `(M, 1)` with `requires_grad=True`). Use `pinnode.derivative(y, t)` for $y_\theta'$.

In [ ]:
def residual(model, t):
    y = model(t)
    # TODO: return the residual y' + 2 t y (same shape as y)
    raise NotImplementedError

**Check your answer.** The exact solution, wrapped as a 'model', must have a residual of the order of rounding errors.

In [ ]:
exact_model = lambda t: torch.exp(-t ** 2)
t_chk = to_tensor(np.linspace(0.0, T, 50), requires_grad=True)
res = residual(exact_model, t_chk).detach().abs().max().item()
print(f"max |residual of exact solution| = {res:.2e}")
assert res < 1e-12, "the residual of the exact solution should vanish"
print("Task 1 passed.")

## Task 2: soft and hard initial conditions

1. Complete `make_loss(model, hard)` returning a closure `loss_fn()` that gives `(loss, parts)` as required by `pinnode.train`
   (see the listing in Section 1.5). Use 100 uniformly spaced collocation points and $\lambda_{ic}=1$.
2. Train both variants with `train(model.parameters(), loss_fn, adam_iters=3000, lr=1e-3, lr_final=1e-5, lbfgs_iters=300)`
   for a network `MLP(1, 1, width=32, depth=2, activation="tanh", domain=(0.0, T))`.
3. Report the relative $L^2$ and maximum errors on `t_test`.

In [ ]:
t_col = to_tensor(np.linspace(0.0, T, 100), requires_grad=True)
t0 = to_tensor([0.0])

def make_loss(model, hard):
    def loss_fn():
        r = residual(model, t_col)
        L_r = (r ** 2).mean()
        if hard:
            return L_r, {"residual": L_r}
        # TODO: add the initial-condition term (model(t0) - 1)^2 with weight 1 and return
        #       (total loss, {"residual": L_r, "ic": L_ic})
        raise NotImplementedError
    return loss_fn

results = {}
for hard in (False, True):
    torch.manual_seed(1234)
    net = MLP(1, 1, width=32, depth=2, activation="tanh", domain=(0.0, T))
    # TODO: for hard=True wrap the network in ConstrainedModel with the transform 1 + t * n
    model = net
    hist = train(model.parameters(), make_loss(model, hard), adam_iters=3000, lr=1e-3,
                 lr_final=1e-5, lbfgs_iters=300, print_every=1000)
    with torch.no_grad():
        y_pred = model(to_tensor(t_test)).numpy().ravel()
    results["hard" if hard else "soft"] = (rel_l2(y_pred, y_exact), linf(y_pred, y_exact))
    print("hard" if hard else "soft", "relative L2 = %.2e, max error = %.2e" % results["hard" if hard else "soft"])

**Check your answer.** Both formulations should reach a relative error well below $10^{-2}$; typically the errors are between $10^{-6}$ and $10^{-3}$.

In [ ]:
for k, (e2, einf) in results.items():
    assert e2 < 1e-2, f"{k}: relative L2 error {e2:.2e} is too large - check the loss"
print("Task 2 passed:", {k: f"{v[0]:.1e}" for k, v in results.items()})

## Task 3: variance of the Monte Carlo loss

For the trained soft network, draw $M\in\{25, 100, 400\}$ independent uniform points on $[0,2]$, evaluate $\mathcal{L}_r$ for 200 independent draws each,
and compare the empirical variance with the formula of the proposition (evaluate the integrals on 4001 points with the trapezoidal rule).
How does the standard deviation scale with $M$?

In [ ]:
rng = np.random.default_rng(0)
# TODO: (1) evaluate r on a fine grid and compute I2 = mean of r^2 and I4 = mean of r^4 (trapezoidal rule / T)
#       (2) for each M, draw 200 samples of the loss and compare np.var(samples) with (I4 - I2**2) / M


## Task 4: a fair comparison with a classical solver

Solve the problem with `solve_ivp(..., method="RK45")` for `rtol` in `1e-3, ..., 1e-10` (with `atol = rtol * 1e-3`), record the maximum error on `t_test`
(use `dense_output=True`) and the number of right-hand-side evaluations `sol.nfev`. Which tolerance matches the accuracy of your PINN, and how many
evaluations does it need? Compare with the number of network evaluations used during training (iterations $\times$ collocation points).

In [ ]:
f = lambda t, y: -2.0 * t * y
# TODO: loop over tolerances, call solve_ivp, and print rtol, nfev and the maximum error


## Discussion questions
* Why can the soft formulation have a larger error near $t=0$ than the hard one?
* In Task 3, does the variance formula still apply to the *trained* network if the same random points were used for training? (See Section 2.5.)
* Which of the uses of PINNs listed in Section 1.3 would justify the extra cost observed in Task 4?